# 03 — Формування панельного датасету ВВП × нічне освітлення

Ноутбук об'єднує панель нічного освітлення з ноутбука 02 з реальним ВВП Eurostat, додає похідні змінні для моделей рівнів і динаміки та зберігає фінальну панель «країна — рік» за 2012–2025 роки.

**Етапи:**
1. Налаштування середовища та параметрів.
2. Завантаження панелі світла і ВВП.
3. Злиття та перевірка узгодженості.
4. Похідні змінні.
5. Контроль якості та збереження.

**Вхідні файли:** `data/processed/ntl/ntl_panel_2012_2025.csv` (ноутбук 02), `data/economic/eurostat_nama_10_gdp_2012_2025.csv` (ноутбук 01).

**Вихідний файл:** `data/processed/gdp_ntl_panel_2012_2025.csv`.

**Принципи:**
- ВВП береться лише в **реальних** одиницях (`CLV10_MEUR`, ланцюгові обсяги в цінах 2010 року, показник `B1GQ`). Номінальний ВВП містить інфляцію, яка не пов'язана зі світлом.
- Країни без ВВП (`LI`) і з неповним рядом (`UK`) у панель не потрапляють, і це перевіряється.
- Набір ознак світла фіксується **заздалегідь**, без відбору за кореляцією з ВВП, щоб уникнути зсуву відбору.

## 1. Налаштування середовища та параметрів

Підключення Google Drive, імпорти, шляхи проєкту і параметри. Поза Colab корінь проєкту береться зі змінної середовища `PROJECT_ROOT`.

- `GDP_UNIT`, `GDP_ITEM` — одиниця виміру і показник ВВП, які залишаються з файлу Eurostat;
- `EXCLUDED_COUNTRIES` — країни, яких не має бути в панелі;
- `LOG_SOURCES` — «сумарні» ознаки світла, для яких рахуються логарифми;
- `GROWTH_SOURCES` — ознаки, для яких рахуються річні прирости логарифмів (усі, крім площі маски, бо вона стала в часі).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/GDP_NTL_Project")
except ImportError:                       # запуск поза Colab
    PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "."))

NTL_PANEL_PATH = PROJECT_ROOT / "data" / "processed" / "ntl" / "ntl_panel_2012_2025.csv"
GDP_PATH = PROJECT_ROOT / "data" / "economic" / "eurostat_nama_10_gdp_2012_2025.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "gdp_ntl_panel_2012_2025.csv"

GDP_UNIT = "CLV10_MEUR"
GDP_ITEM = "B1GQ"
YEARS = list(range(2012, 2026))
N_COUNTRIES = 35
EXCLUDED_COUNTRIES = {"LI", "UK"}

LOG_SOURCES = ["ntl_sum_area_weighted", "ntl_sum_cap50", "ntl_sum_cap100", "ntl_sum_log1p",
               "lit_area_km2_t1", "lit_area_km2_t2", "lit_area_km2_t5", "total_mask_km2"]
GROWTH_SOURCES = [c for c in LOG_SOURCES if c != "total_mask_km2"]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Завантаження панелі світла і ВВП

- `load_ntl` — читає панель світла з ноутбука 02 і перевіряє, що пари «країна × рік» унікальні і що є всі ознаки з `LOG_SOURCES`;
- `load_gdp` — читає таблицю Eurostat у «широкому» форматі (рік — окрема колонка), залишає одну одиницю виміру й один показник і переводить таблицю в довгий формат `CNTR_ID — year — gdp_meur`.

In [ ]:
def load_ntl(path=NTL_PANEL_PATH):
    ntl = pd.read_csv(path)
    missing = set(LOG_SOURCES) - set(ntl.columns)
    assert not missing, f"У панелі світла немає колонок: {sorted(missing)}"
    assert not ntl.duplicated(["CNTR_ID", "year"]).any(), "Дублікати країна-рік у панелі світла"
    print(f"NTL: {ntl.shape} | країн: {ntl['CNTR_ID'].nunique()} | "
          f"роки: {ntl['year'].min()}–{ntl['year'].max()}")
    return ntl


def load_gdp(path=GDP_PATH):
    raw = pd.read_csv(path)
    if "unit" in raw.columns:
        print("Одиниці у файлі:", raw["unit"].unique().tolist())
        raw = raw[raw["unit"] == GDP_UNIT]
    if "na_item" in raw.columns:
        raw = raw[raw["na_item"] == GDP_ITEM]
    assert len(raw), f"У файлі немає {GDP_UNIT}/{GDP_ITEM}"

    geo_col = next(c for c in raw.columns if c.startswith("geo"))
    year_cols = [c for c in raw.columns if str(c).strip().isdigit()]
    gdp = (raw.melt(id_vars=[geo_col], value_vars=year_cols, var_name="year", value_name="gdp_meur")
              .rename(columns={geo_col: "CNTR_ID"}))
    gdp["year"] = gdp["year"].astype(int)
    gdp["gdp_meur"] = pd.to_numeric(gdp["gdp_meur"], errors="coerce")
    gdp = gdp[gdp["year"].isin(YEARS)]
    assert not gdp.duplicated(["CNTR_ID", "year"]).any(), "Дублікати країна-рік у ВВП"
    print(f"GDP: {gdp.shape} | країн: {gdp['CNTR_ID'].nunique()}")
    return gdp


ntl = load_ntl()
gdp = load_gdp()

NTL: (490, 53) | країн: 35 | роки: 2012–2025
GDP: (490, 3) | країн: 35


## 3. Злиття та перевірка узгодженості

- `merge_panel` — об'єднує світло і ВВП (inner join) і відкидає роки без ВВП. Функція перевіряє, що:
  - для кожної країни зі світлом є ВВП (інакше, ймовірно, коди країн не збігаються, наприклад `EL`/`GR`);
  - ряд кожної країни не має пропусків усередині, бо від цього залежать прирости в розділі 4.

Таблиця показує країни з неповним рядом (зараз це лише `AL`, для якої немає ВВП за 2025 рік).

In [ ]:
def merge_panel(ntl, gdp):
    only_ntl = sorted(set(ntl["CNTR_ID"]) - set(gdp["CNTR_ID"]))
    only_gdp = sorted(set(gdp["CNTR_ID"]) - set(ntl["CNTR_ID"]))
    print("Є світло, немає ВВП:", only_ntl, "(очікується: [])")
    print("Є ВВП, немає світла:", only_gdp, "(очікується: ['UK'] — свідомо виключена)")
    assert not only_ntl, "Невідповідність кодів країн (наприклад, EL/GR)"

    df = (ntl.merge(gdp, on=["CNTR_ID", "year"], how="inner")
             .dropna(subset=["gdp_meur"])
             .sort_values(["CNTR_ID", "year"])
             .reset_index(drop=True))

    gaps = df.groupby("CNTR_ID")["year"].apply(lambda s: (s.diff().dropna() != 1).any())
    assert not gaps.any(), f"Пропуски всередині ряду: {gaps[gaps].index.tolist()}"

    cover = df.groupby("CNTR_ID")["year"].agg(["min", "max", "count"])
    display(cover[cover["count"] < len(YEARS)])
    print(f"Панель: {len(df)} спостережень, {df['CNTR_ID'].nunique()} країн")
    return df


df = merge_panel(ntl, gdp)

Є світло, немає ВВП: [] (очікується: [])
Є ВВП, немає світла: [] (очікується: ['UK'] — свідомо виключена)


,min,max,count
CNTR_ID,,,
AL,2012,2024,13


Панель: 489 спостережень, 35 країн


## 4. Похідні змінні

- `safe_log` — логарифм, де неположні значення стають NaN;
- `add_features` — додає до панелі:
  - `ln_gdp` — логарифм ВВП (ціль для моделей рівнів);
  - `ln_<ознака>` — логарифми ознак з `LOG_SOURCES`;
  - `lit_share_t2` — частку площі маски країни зі світлом ≥ 2 nW/(cm²·sr);
  - `dlgdp`, `d_ln_<ознака>` — річні прирости логарифмів (ціль і ознаки для моделей динаміки);
  - `ln_gdp_lag1` — лаг логарифма ВВП.

Прирости рахуються всередині країни. Це коректно, бо ряд відсортований і не має пропусків (перевірено в розділі 3).

In [ ]:
def safe_log(s):
    return np.log(s.where(s > 0))


def add_features(df):
    df = df.copy()
    by_country = df.groupby("CNTR_ID")

    df["ln_gdp"] = np.log(df["gdp_meur"])
    for c in LOG_SOURCES:
        df[f"ln_{c}"] = safe_log(df[c])
    df["lit_share_t2"] = df["lit_area_km2_t2"] / df["total_mask_km2"]

    df["dlgdp"] = by_country["ln_gdp"].diff()
    for c in GROWTH_SOURCES:
        df[f"d_ln_{c}"] = by_country[f"ln_{c}"].diff()
    df["ln_gdp_lag1"] = by_country["ln_gdp"].shift(1)
    return df


df = add_features(df)

log_cols = ["ln_gdp"] + [f"ln_{c}" for c in LOG_SOURCES]
n_nan = df[log_cols].isna().sum()
print("NaN у логарифмах:", n_nan[n_nan > 0].to_dict() or "немає")

NaN у логарифмах: немає


## 5. Контроль якості та збереження

- `validate_panel` — фінальні перевірки панелі:
  - рівно 35 країн, серед них немає `LI` і `UK`;
  - усі логарифми визначені (ВВП і ознаки світла додатні);
  - річні прирости ВВП правдоподібні (|Δln| < 0.5, тобто зміна менша ніж ~40 %).

Панель зберігається в `gdp_ntl_panel_2012_2025.csv`, лише якщо всі перевірки пройдено.

In [ ]:
def validate_panel(df):
    checks = {
        f"країн = {N_COUNTRIES}":        df["CNTR_ID"].nunique() == N_COUNTRIES,
        "немає LI та UK":               not EXCLUDED_COUNTRIES & set(df["CNTR_ID"]),
        "логарифми без NaN":            bool(df[log_cols].notna().all().all()),
        "прирости (|Δln|<0.5)": bool((df["dlgdp"].abs().dropna() < 0.5).all()),
    }
    display(pd.Series(checks, name="OK"))
    assert all(checks.values()), "Є непройдені перевірки"


validate_panel(df)
df.to_csv(OUTPUT_PATH, index=False)
print("Збережено:", OUTPUT_PATH, df.shape)
display(df[["CNTR_ID", "year", "gdp_meur", "ln_gdp", "dlgdp", "ntl_sum_cap50", "ln_ntl_sum_cap50"]].head())

,OK
країн = 35,True
немає LI та UK,True
логарифми без NaN,True
прирости (|Δln|<0.5),True


Збережено: /content/drive/MyDrive/GDP_NTL_Project/data/processed/gdp_ntl_panel_2012_2025.csv (489, 73)


,CNTR_ID,year,gdp_meur,ln_gdp,dlgdp,ntl_sum_cap50,ln_ntl_sum_cap50
0,AL,2012,9433.6,9.152033,NaN,12670.243817,9.447012
1,AL,2013,9594.6,9.168956,0.016923,12443.342495,9.428941
2,AL,2014,9809.6,9.191117,0.022161,11907.883003,9.384956
3,AL,2015,10028.1,9.213146,0.022030,11121.089179,9.316599
4,AL,2016,10420.1,9.251492,0.038345,10506.693492,9.259768
